# Example 01: 3D Cs-137 Volumetric Activity Reconstruction

Reconstruct volumetric Cs-137 activity in soil from simulated dose-rate measurements.

**Workflow:** synthetic 3D source -> forward dose rates -> MLEM/Tikhonov -> SAD (Fredholm) -> 3D plots (matplotlib + Plotly).

Refs: Chizhov et al (2019, 2023) JRP; ICRP 74.


## 1. Imports and parameters

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

from soilactivity import (
    Unfolder, UnfoldingResult,
    SadReconstructor, SadResult,
    KERMA_CONSTANTS,
    h_star_10_over_phil,
    lorenz_gini_coefficient,
)

print('soilactivity: ok')
print('Cs-137 kerma:', KERMA_CONSTANTS['Cs-137'], 'aGy m^2/(s Bq)')

In [ ]:
SOIL_DENSITY = 1.6
MU_SOIL = 0.12
X_RANGE = (0.0, 50.0)
Y_RANGE = (0.0, 50.0)
Z_RANGE = (0.0, -0.50)
NX, NY, NZ = 12, 12, 8
np.random.seed(42)

## 2. Synthetic 3D Cs-137 source (two Gaussian hot-spots)

In [ ]:
grid_x = np.linspace(*X_RANGE, NX+1)
grid_y = np.linspace(*Y_RANGE, NY+1)
grid_z = np.linspace(*Z_RANGE, NZ+1)
cx = 0.5*(grid_x[:-1]+grid_x[1:])
cy = 0.5*(grid_y[:-1]+grid_y[1:])
cz = 0.5*(grid_z[:-1]+grid_z[1:])
CX, CY, CZ = np.meshgrid(cx, cy, cz, indexing='ij')
depths_cm = abs(cz)*100
print(f'Grid: {NX}x{NY}x{NZ} = {NX*NY*NZ} voxels')
print('Depths:', [f'{d:.1f}cm' for d in depths_cm])

In [ ]:
activity_true = np.zeros((NX,NY,NZ))
x0_1,y0_1,z0_1,sx_1,sy_1,sz_1 = 22.,25.,-0.05,6.,7.,0.08
A1 = 5e5
x0_2,y0_2,z0_2,sx_2,sy_2,sz_2 = 35.,15.,-0.25,5.,5.,0.10
A2 = 2e5
for i in range(NX):
    for j in range(NY):
        for k in range(NZ):
            x,y,z = CX[i,j,k],CY[i,j,k],CZ[i,j,k]
            g1 = A1*np.exp(-((x-x0_1)**2/(2*sx_1**2)+(y-y0_1)**2/(2*sy_1**2)+(z-z0_1)**2/(2*sz_1**2)))
            g2 = A2*np.exp(-((x-x0_2)**2/(2*sx_2**2)+(y-y0_2)**2/(2*sy_2**2)+(z-z0_2)**2/(2*sz_2**2)))
            activity_true[i,j,k] = (g1+g2)*np.exp(-abs(z)*100/15.)
total_Bq = np.sum(activity_true)
print(f'Total: {total_Bq:.3e} Bq ({total_Bq/1e6:.2f} MBq)')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('True Cs-137 Activity -- Depth Slices', fontsize=14, fontweight='bold')
vmax = np.max(activity_true)
for k in range(NZ):
    ax = axes[k//4, k%4]
    im = ax.pcolormesh(cx, cy, activity_true[:,:,k].T, shading='auto', cmap='YlOrRd', vmin=0, vmax=vmax)
    ax.set_title(f'z = {depths_cm[k]:.1f} cm')
    ax.set_xlabel('X, m'); ax.set_ylabel('Y, m')
    ax.set_aspect('equal')
    plt.colorbar(im, ax=ax, label='Bq/voxel')
plt.tight_layout(); plt.show()

## 3. Forward problem: simulate dose-rate measurements

In [ ]:
n_meas = 200
meas_x = np.random.uniform(X_RANGE[0]+2, X_RANGE[1]-2, n_meas)
meas_y = np.random.uniform(Y_RANGE[0]+2, Y_RANGE[1]-2, n_meas)
DET_H = 1.0
h10_phi = h_star_10_over_phil(0.662)
dose_rates = np.zeros(n_meas)

for m in range(n_meas):
    det = np.array([meas_x[m], meas_y[m], DET_H])
    tot = 0.0
    for i in range(NX):
        for j in range(NY):
            for k in range(NZ):
                a = activity_true[i,j,k]
                if a < 1.: continue
                src = np.array([CX[i,j,k], CY[i,j,k], CZ[i,j,k]])
                r = max(np.linalg.norm(det - src), 0.01)
                r_cm = r * 100.
                soil_path = abs(CZ[i,j,k]) * 100.
                attn = np.exp(-MU_SOIL * soil_path)
                tot += a * 0.851 * h10_phi * attn / (4*np.pi*r_cm**2)
    dose_rates[m] = tot * 1e6 * 3600  # Sv/s -> uSv/h

print(f'Dose rate: {np.min(dose_rates):.3f} -- {np.max(dose_rates):.3f} uSv/h')
print(f'Mean: {np.mean(dose_rates):.3f} uSv/h')

In [ ]:
NOISE = 0.05
dose_noisy = dose_rates * (1. + NOISE*np.random.randn(n_meas))
dose_noisy = np.maximum(dose_noisy, 0.)

measurements = pd.DataFrame({
    'x': meas_x, 'y': meas_y, 'z': np.zeros(n_meas), 'dose_rate': dose_noisy
})

fig, ax = plt.subplots(figsize=(8,7))
sc = ax.scatter(measurements['x'], measurements['y'], c=measurements['dose_rate'],
               cmap='hot_r', s=40, edgecolors='k', linewidth=0.3)
plt.colorbar(sc, ax=ax, label='uSv/h')
ax.set_xlabel('X, m'); ax.set_ylabel('Y, m')
ax.set_title('Simulated Dose-Rate Measurements (z=0)')
ax.set_aspect('equal'); plt.tight_layout(); plt.show()

## 4. 3D Volumetric Unfolding (MLEM)

In [ ]:
grids = (grid_x, grid_y, grid_z)

unf_mlem = Unfolder(method='mlem', iterations=80, tol=1e-5)
res_mlem = unf_mlem.unfold(measurements, grids=grids,
    attenuation_coeff=MU_SOIL/100., smooth_sigma=0.5)

print(f'Iterations: {res_mlem.solver_info["iterations"]}')
print(f'Converged: {res_mlem.solver_info["converged"]}')
print(f'Residual: {res_mlem.solver_info["final_residual"]:.4e}')
print(f'Recon total: {np.sum(res_mlem.activity_3d):.3e} Bq ({np.sum(res_mlem.activity_3d)/1e6:.2f} MBq)')
print(f'True total:  {total_Bq:.3e} Bq ({total_Bq/1e6:.2f} MBq)')

In [ ]:
unf_tikh = Unfolder(method='tikhonov', lambda_reg=1e-3)
res_tikh = unf_tikh.unfold(measurements, grids=grids,
    attenuation_coeff=MU_SOIL/100., smooth_sigma=0.5)

print(f'Tikhonov converged: {res_tikh.solver_info["converged"]}')
print(f'Tikhonov total: {np.sum(res_tikh.activity_3d):.3e} Bq ({np.sum(res_tikh.activity_3d)/1e6:.2f} MBq)')

In [ ]:
fig, axes = plt.subplots(3, NZ, figsize=(22, 14))
fig.suptitle('Cs-137 3D Reconstruction -- MLEM vs True', fontsize=15, fontweight='bold')
vmax_c = max(np.max(activity_true), np.max(res_mlem.activity_3d))

for k in range(NZ):
    for row, (data, label) in enumerate([
        (activity_true, 'True'), (res_mlem.activity_3d, 'MLEM'),
        (res_mlem.activity_3d - activity_true, 'Diff')]):
        ax = axes[row, k]
        if row < 2:
            im = ax.pcolormesh(cx, cy, data[:,:,k].T, shading='auto',
                cmap='YlOrRd', vmin=0, vmax=vmax_c)
        else:
            vd = max(abs(data[:,:,k].min()), abs(data[:,:,k].max()))
            im = ax.pcolormesh(cx, cy, data[:,:,k].T, shading='auto',
                cmap='RdBu_r', vmin=-vd, vmax=vd)
        ax.set_title(f'{label} | z={depths_cm[k]:.1f}cm', fontsize=9)
        ax.set_aspect('equal')
        plt.colorbar(im, ax=ax)
for r in range(3): axes[r,0].set_ylabel('Y, m')
for k in range(NZ): axes[2,k].set_xlabel('X, m')
plt.tight_layout(); plt.show()

## 5. 3D Surface Plot -- Matplotlib

In [ ]:
X_mesh, Y_mesh = np.meshgrid(cx, cy, indexing='ij')Z_true = activity_true[:,:,0].TZ_recon = res_mlem.activity_3d[:,:,0].Tfig = plt.figure(figsize=(14, 6))ax1 = fig.add_subplot(121, projection='3d')ax1.plot_surface(X_mesh, Y_mesh, Z_true, cmap='YlOrRd', alpha=0.9,    rstride=1, cstride=1, edgecolor='k', linewidth=0.2)ax1.set_xlabel('X, m'); ax1.set_ylabel('Y, m'); ax1.set_zlabel('Activity, Bq')ax1.set_title(f'True (z={depths_cm[0]:.1f} cm)', fontweight='bold')fig.colorbar(ax1.collections[0], ax=ax1, shrink=0.5, label='Bq/voxel')ax2 = fig.add_subplot(122, projection='3d')ax2.plot_surface(X_mesh, Y_mesh, Z_recon, cmap='YlOrRd', alpha=0.9,    rstride=1, cstride=1, edgecolor='k', linewidth=0.2)ax2.set_xlabel('X, m'); ax2.set_ylabel('Y, m'); ax2.set_zlabel('Activity, Bq')ax2.set_title(f'MLEM Recon (z={depths_cm[0]:.1f} cm)', fontweight='bold')fig.colorbar(ax2.collections[0], ax=ax2, shrink=0.5, label='Bq/voxel')plt.tight_layout(); plt.show()

In [ ]:
fig = plt.figure(figsize=(16, 10))ax = fig.add_subplot(111, projection='3d')depth_colors = cm.viridis(np.linspace(0.1, 0.9, NZ))for k in range(NZ):    Z_surf = res_mlem.activity_3d[:,:,k].T    z_off = abs(cz[k]) * 100    Z_norm = Z_surf / (np.max(Z_surf)+1e-10) * 2.0    ax.plot_surface(X_mesh, Y_mesh, Z_norm + z_off,        color=depth_colors[k], alpha=0.5, rstride=1, cstride=1,        edgecolor='gray', linewidth=0.15)ax.set_xlabel('X, m', fontsize=11)ax.set_ylabel('Y, m', fontsize=11)ax.set_zlabel('Depth / Activity', fontsize=11)ax.set_title('3D Cs-137 Activity -- All Depth Layers (MLEM)', fontsize=13, fontweight='bold')ax.view_init(elev=25, azim=-60)for k in range(NZ):    ax.text(52, 52, abs(cz[k])*100, f'{depths_cm[k]:.1f}cm', fontsize=8, color=depth_colors[k])plt.tight_layout(); plt.show()

## 6. Interactive 3D Plot -- Plotly

In [ ]:
import plotly.graph_objects as gofrom plotly.subplots import make_subplotsfig = make_subplots(rows=1, cols=2,    specs=[[{'type': 'surface'}, {'type': 'surface'}]],    horizontal_spacing=0.05,    subplot_titles=(f'True Cs-137 (z={depths_cm[0]:.1f}cm)', f'MLEM Recon (z={depths_cm[0]:.1f}cm)'))fig.add_trace(go.Surface(x=cx, y=cy, z=Z_true, colorscale='YlOrRd',    colorbar=dict(title='Bq/voxel', x=0.45), name='True'), row=1, col=1)fig.add_trace(go.Surface(x=cx, y=cy, z=Z_recon, colorscale='YlOrRd',    colorbar=dict(title='Bq/voxel', x=1.0), name='Recon'), row=1, col=2)fig.update_layout(title='Cs-137 3D Reconstruction -- Interactive', width=1200, height=600,    scene=dict(xaxis_title='X, m', yaxis_title='Y, m', zaxis_title='Activity, Bq'),    scene2=dict(xaxis_title='X, m', yaxis_title='Y, m', zaxis_title='Activity, Bq'))fig.show()

In [ ]:
fig_all = go.Figure()cscales = ['Viridis','Plasma','Inferno','YlOrRd','Hot','Reds','Oranges','YlOrBr']for k in range(NZ):    Z_s = res_mlem.activity_3d[:,:,k].T    z_off = abs(cz[k]) * 100    fig_all.add_trace(go.Surface(x=cx, y=cy, z=Z_s + z_off,        colorscale=cscales[k % len(cscales)], opacity=0.75,        name=f'z={depths_cm[k]:.1f}cm', showscale=(k==0),        colorbar=dict(title='Bq/voxel') if k==0 else None))fig_all.update_layout(title='3D Cs-137 Volumetric -- All Layers (Interactive)', width=1000, height=800,    scene=dict(xaxis_title='X, m', yaxis_title='Y, m', zaxis_title='Depth(cm)+Activity(Bq)',        camera=dict(eye=dict(x=1.8, y=1.8, z=1.2))))fig_all.show()

## 7. 2D Surface Activity Density via Fredholm Equation

In [ ]:
from scipy.interpolate import Rbfnx_2d, ny_2d = 20, 20cell_2d = X_RANGE[1] / nx_2drbf = Rbf(meas_x, meas_y, dose_noisy, function='thin_plate')x2d = np.linspace(cell_2d/2, X_RANGE[1]-cell_2d/2, nx_2d)y2d = np.linspace(cell_2d/2, Y_RANGE[1]-cell_2d/2, ny_2d)X2d, Y2d = np.meshgrid(x2d, y2d, indexing='ij')ader_map = np.maximum(rbf(X2d, Y2d), 0.0)ader_sv_s = ader_map / (1e6 * 3600)print(f'ADER raster: {nx_2d}x{ny_2d}, cell={cell_2d:.1f}m')print(f'ADER range: {np.min(ader_map):.3f} -- {np.max(ader_map):.3f} uSv/h')

In [ ]:
recon = SadReconstructor(nx=nx_2d, ny=ny_2d, cell_size=cell_2d,    height_m=DET_H, radionuclide='Cs-137', dose_quantity='H_star_10')result_sad = recon.reconstruct(ader_sv_s, alpha=1e-12, non_negative=True)print('Method:', result_sad.method)print(f'Alpha: {result_sad.alpha:.2e}')print(f'Total (Fredholm): {result_sad.total_activity:.3e} Bq')print(f'Total (MCC): {result_sad.total_activity_mcc:.3e} Bq')info = result_sad.infoprint(f'Cond(F): {info["cond_F"]:.2e}')print(f'Gini SAD: {info["gini_sad"]:.4f}')print(f'Gini ADER: {info["gini_ader"]:.4f}')print(f'Compactness: {info["compactness_ratio"]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))fig.suptitle('2D SAD Reconstruction (Fredholm)', fontsize=14, fontweight='bold')im1 = axes[0].pcolormesh(x2d, y2d, ader_map.T, shading='auto', cmap='hot_r')axes[0].set_title('ADER input (uSv/h)')axes[0].set_xlabel('X, m'); axes[0].set_ylabel('Y, m'); axes[0].set_aspect('equal')plt.colorbar(im1, ax=axes[0])sad_m2 = result_sad.sad / cell_2d**2im2 = axes[1].pcolormesh(x2d, y2d, sad_m2.T, shading='auto', cmap='YlOrRd')axes[1].set_title('SAD (Bq/m^2)')axes[1].set_xlabel('X, m'); axes[1].set_ylabel('Y, m'); axes[1].set_aspect('equal')plt.colorbar(im2, ax=axes[1])ader_fwd = result_sad.ader_forward * 1e6 * 3600im3 = axes[2].pcolormesh(x2d, y2d, ader_fwd.T, shading='auto', cmap='hot_r')axes[2].set_title('ADER forward (uSv/h)')axes[2].set_xlabel('X, m'); axes[2].set_ylabel('Y, m'); axes[2].set_aspect('equal')plt.colorbar(im3, ax=axes[2])plt.tight_layout(); plt.show()

## 8. Interactive 3D SAD -- Plotly

In [ ]:
fig_sad = go.Figure()fig_sad.add_trace(go.Surface(x=x2d, y=y2d, z=sad_m2.T,    colorscale='YlOrRd', colorbar=dict(title='SAD, Bq/m^2'), name='SAD',    contours=dict(z=dict(show=True, usecolormap=True, highlightcolor='limegreen', project_z=True))))fig_sad.update_layout(title='Cs-137 Surface Activity Density -- Fredholm', width=900, height=700,    scene=dict(xaxis_title='X, m', yaxis_title='Y, m', zaxis_title='SAD, Bq/m^2',        camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))))fig_sad.show()

## 9. MLEM Convergence & Depth Profile

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))hist = res_mlem.solver_info['history']ax1.semilogy(range(1, len(hist)+1), hist, 'b-o', markersize=3)ax1.set_xlabel('Iteration'); ax1.set_ylabel('Rel. change (log)')ax1.set_title('MLEM Convergence')ax1.axhline(1e-5, color='r', ls='--', label='Tol')ax1.legend(); ax1.grid(True, alpha=0.3)true_layer = [np.sum(activity_true[:,:,k]) for k in range(NZ)]recon_layer = [np.sum(res_mlem.activity_3d[:,:,k]) for k in range(NZ)]w = 0.35; xp = np.arange(NZ)ax2.bar(xp-w/2, np.array(true_layer)/1e6, w, label='True', color='steelblue')ax2.bar(xp+w/2, np.array(recon_layer)/1e6, w, label='MLEM', color='coral')ax2.set_xlabel('Depth, cm'); ax2.set_ylabel('Activity, MBq')ax2.set_title('Activity vs Depth')ax2.set_xticks(xp); ax2.set_xticklabels([f'{d:.1f}' for d in depths_cm])ax2.legend(); ax2.grid(axis='y', alpha=0.3)plt.tight_layout(); plt.show()

## 10. Summary

In [ ]:
si = res_mlem.solver_infoprint('='*60)print('Radionuclide:       Cs-137')print(f'Grid:               {NX}x{NY}x{NZ} voxels')print(f'True total:         {total_Bq:.3e} Bq ({total_Bq/1e6:.2f} MBq)')print(f'MLEM recon:         {np.sum(res_mlem.activity_3d):.3e} Bq ({np.sum(res_mlem.activity_3d)/1e6:.2f} MBq)')print(f'Tikhonov recon:     {np.sum(res_tikh.activity_3d):.3e} Bq ({np.sum(res_tikh.activity_3d)/1e6:.2f} MBq)')print('MLEM iterations:   ', si["iterations"])print(f'SAD Fredholm:       {result_sad.total_activity:.3e} Bq ({result_sad.total_activity/1e6:.2f} MBq)')print(f'SAD MCC:            {result_sad.total_activity_mcc:.3e} Bq ({result_sad.total_activity_mcc/1e6:.2f} MBq)')info = result_sad.infoprint(f'Cond(F):            {info["cond_F"]:.2e}')print(f'Gini SAD/ADER:      {info["gini_sad"]:.4f} / {info["gini_ader"]:.4f}')print('='*60)